# Phase 1 Baselines on Colab GPU — DLinear, NLinear, XGBoost, LSTM (CPCB AQI)

**Purpose**: This notebook reproduces the Phase 1 baseline model comparison from
`Implementation/phase1_baselines/` on a Google Colab GPU runtime, instead of the
author's slow local CPU machine. It is a faithful, self-contained transcription
(not a rewrite) of the following source files, kept for traceability:

| Source file | What it contributes here |
|---|---|
| `phase1_baselines/config_phase1.py` | Feature/target columns, split ratios, hyperparameters, random seed |
| `phase1_baselines/comparison/data_utils.py` | `AQIDataLoader` (cleaning + sequence windowing) and `XGBoostDataProcessor` (lag features) |
| `phase1_baselines/comparison/evaluate.py` | `Evaluator` / `ComparisonReport` (RMSE, MAE, MAPE, R²) |
| `phase1_baselines/models/dlinear.py` | `DecompositionBlock`, `DLinear`, `DLinearTrainer` |
| `phase1_baselines/models/nlinear.py` | `NLinear`, `NLinearTrainer` |
| `phase1_baselines/models/xgboost_model.py` | `XGBoostModel`, `create_xgboost_model` |
| `phase1_baselines/models/lstm_wrapper.py` | `LSTMTrainer` (Keras/TensorFlow LSTM baseline) |
| `phase1_baselines/train_dlinear_only.py`, `train_nlinear_only.py`, `train_xgboost_only.py` | Per-model training-loop logic (per lookback period) |
| `phase1_baselines/compare_lookback_comprehensive.py` | The training/evaluation loop that trains **all four** models (DLinear, NLinear, XGBoost, LSTM) across lookback periods 7/14/30 — the roster this notebook reproduces |
| `phase1_baselines/run_comparison.py`, `comparison/compare_all_refactored.py` | Final aggregation/report/plot logic |

**Dataset**: CPCB India city-day air quality data (`city_day.csv`, ~2.5 MB, 26 cities).
Target: `AQI`. Features: 12 pollutant concentrations.

**What changed vs. the original scripts** (see the "Deviations from source" note near the
end for the full list): device is explicitly detected and wired into every model
constructor that accepts one (the original scripts hard-code `device='cpu'` and never
override it), and a global random seed is applied everywhere it wasn't before. Everything
else — architectures, preprocessing, hyperparameters, split logic, metric formulas — is
transcribed as-is.


# 🔖 Manuscript <-> Notebook cross-reference

This notebook backs the baseline rows of the manuscript *"Interpretable temporal attribution for
city-specific air-quality forecasting."* It trains the four Phase-1 baselines (DLinear, NLinear,
XGBoost, LSTM) under a strictly causal evaluation protocol and produces the DLinear / NLinear / XGBoost
values reported in **Table 2** (`tab:main_results`). Each result section carries a **📄 Maps to
manuscript** tag. Table/figure numbers are current as of the committed draft; the LaTeX labels (in
`typewriter`) stay valid if numbering shifts.

### A. Notebook -> Manuscript

| Notebook section | Produces | Manuscript element |
|---|---|---|
| **1** Setup + seeding | env, `RANDOM_SEED` | Methods *Reproducibility* / *Implementation details* |
| **2** Dataset | CPCB `city_day.csv` load | Methods *CPCB India dataset* |
| **3-4** Config + preprocessing | per-city clean, min-max, chronological split, lag features | Methods *Preprocessing*, *Train/validation/test split* (70/10/20), *Baseline forecasting models* |
| **5** DLinear | R2=0.795 @ LB7 | Table 2 (`tab:main_results`) DLinear row; Methods *DLinear/NLinear* |
| **6** NLinear | R2=0.807 @ LB7 | Table 2 (`tab:main_results`) NLinear row; Methods *DLinear/NLinear* |
| **7** XGBoost | R2=0.864 @ LB7 (best on every metric) | Table 2 (`tab:main_results`) XGBoost row; Methods *XGBoost* |
| **8** LSTM (Keras) | R2=0.813 @ LB7 | This notebook's own LSTM baseline; the manuscript's Table 2 LSTM row (0.818) is the PyTorch LSTM from `phase4_advanced_v2_colab_4.ipynb` Section 7c |
| **9-10** Train/eval + comparison | assembled metrics across LB 7/14/30 | Table 2 (`tab:main_results`) baseline column |
| **11** Reference metrics | final-number cross-check | cross-check only |

### B. Manuscript -> Notebook

| Manuscript element | Backed by |
|---|---|
| Table 2 (`tab:main_results`) DLinear / NLinear / XGBoost rows | 5 / 6 / 7 + 9-10 assembly |
| Methods *DLinear and NLinear* | 5, 6 |
| Methods *XGBoost* | 7 |
| Methods *Preprocessing* + *Train/validation/test split* | 3, 4 |

> Sibling notebook `phase4_advanced_v2_colab_4_paper_mapped.ipynb` backs the proposed model, the
> plain-iTransformer / CEEMDAN+LSTM / LSTM baselines, the SHAP attribution, the ablation, and the
> multi-seed table. Together the two notebooks cover every numerical claim in the paper.


## 1. Setup

### 1.1 Package installs

Colab's runtime already ships GPU-linked builds of `torch`, `tensorflow`, `numpy`,
`pandas`, `scikit-learn`, and `matplotlib` — we deliberately do **not** pip-install or
upgrade any of these, since reinstalling risks breaking the pre-linked CUDA wiring.

The only package this phase imports that might be missing (or present in an older
version) on a given Colab image is `xgboost`. The cell below installs it **only if it
isn't already importable** — no forced reinstall.


In [1]:
import importlib.util

if importlib.util.find_spec("xgboost") is None:
    print("xgboost not found — installing...")
    %pip install -q xgboost
else:
    print("xgboost already available — skipping install (per project constraint: "
          "never reinstall preinstalled GPU-linked packages).")


xgboost already available — skipping install (per project constraint: never reinstall preinstalled GPU-linked packages).


### 1.2 Imports


In [2]:
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # Colab renders inline regardless; avoids backend surprises
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

print("torch:", torch.__version__)
print("tensorflow:", tf.__version__)
print("xgboost:", xgb.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


torch: 2.11.0+cpu
tensorflow: 2.20.0
xgboost: 3.2.0
pandas: 3.0.1
numpy: 2.4.2


### 1.3 GPU detection

`DLinearTrainer` / `NLinearTrainer` (PyTorch) accept a `device` constructor argument
that defaults to `'cpu'` and — in the original `train_*.py` / `compare_lookback_comprehensive.py`
scripts — is **never overridden**, so those scripts train on CPU even when a GPU is
present. Here we detect the device once and pass it explicitly into every PyTorch model
we construct below.

For the Keras/TensorFlow LSTM baseline, TensorFlow uses the GPU automatically if one is
visible to it (no explicit device argument needed) — the print below just confirms
whether TF can see the Colab GPU.


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[PyTorch] Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

print("\n[TensorFlow] Visible GPU devices:", tf.config.list_physical_devices("GPU"))


[PyTorch] Using device: cpu

[TensorFlow] Visible GPU devices: []


### 1.4 Random seeding for reproducibility

Every stochastic component is seeded from a single `RANDOM_SEED` so the run is deterministic
and reproducible: numpy, Python's `random`, PyTorch (CPU + CUDA), TensorFlow/Keras, and
XGBoost's `random_state`. The train/val/test split is chronological (index-sliced, not
shuffled), so it carries no split-level randomness; the `DataLoader`s run with `shuffle=False`.

For full GPU determinism the seeding cell below also sets
`torch.backends.cudnn.deterministic = True` / `benchmark = False` and calls
`tf.config.experimental.enable_op_determinism()`, since cuDNN's default convolution/pooling
and TensorFlow's cuDNN LSTM kernels otherwise select non-deterministic algorithms regardless
of the seed.

---
📄 **Maps to manuscript:** Methods *Reproducibility* / *Implementation details* -- the global seed wired in here makes the baseline run reproducible.

In [4]:
RANDOM_SEED = 42  # from config_phase1.py

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)  # no-op if no GPU

# Deterministic cuDNN kernels -- without this, AvgPool1d/backward on GPU can pick a
# different (faster but non-deterministic) algorithm on each run, even with the seed
# above fixed. Same fix already applied in phase4_advanced_v2.ipynb.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

tf.random.set_seed(RANDOM_SEED)
# TF/Keras's cuDNN-backed LSTM kernel is non-deterministic on GPU by default (atomic-add
# reductions in its backward pass), even with tf.random.set_seed set. This forces the
# deterministic kernel variants instead.
tf.config.experimental.enable_op_determinism()

print(f"Seeded numpy, random, torch (CPU+CUDA), and tensorflow with RANDOM_SEED={RANDOM_SEED}")
print("Enabled cuDNN determinism (torch) and op determinism (tensorflow).")


Seeded numpy, random, torch (CPU+CUDA), and tensorflow with RANDOM_SEED=42
Enabled cuDNN determinism (torch) and op determinism (tensorflow).


## 2. Dataset

Upload `city_day.csv` (found locally at
`Implementation\\Datasets\\kaggle\\2\\city_day.csv`) when prompted by the cell below.

The helper function also checks a few common alternate locations first, so re-running
this cell after the first successful upload (or if you've mounted Google Drive / already
placed the file at `/content/city_day.csv`) won't prompt again.

---
📄 **Maps to manuscript:** Methods *CPCB India dataset* -- five and a half years of daily observations across the 20 retained cities; target is next-day AQI.

In [5]:
def locate_or_upload_csv(filename="city_day.csv"):
    """Return a Path to `filename`, uploading it via the Colab file picker if needed.

    Search order:
      1. Current working directory (e.g. re-running this cell after a prior upload)
      2. /content/<filename>                       (default Colab upload target)
      3. /content/drive/MyDrive/<filename>          (if Google Drive is mounted)
      4. Fall back to files.upload() (Colab file picker)
    """
    candidates = [
        Path(filename),
        Path("/content") / filename,
        Path("/content/drive/MyDrive") / filename,
        # Local (non-Colab) fallbacks -- covers running this notebook from a local
        # Jupyter/VS Code kernel with cwd at the repo root or at this notebook's own
        # folder (Implementation/colab_notebooks/). Matches the same fallback already
        # present in phase3_transformers.ipynb.
        Path("Implementation/Datasets/kaggle/2") / filename,
        Path("../Datasets/kaggle/2") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            print(f"Found dataset at: {candidate.resolve()}")
            return candidate

    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            f"'{filename}' not found locally and google.colab.files is unavailable "
            f"(not running in Colab). Place the file at one of: "
            f"{[str(c) for c in candidates]}"
        )

    print(f"'{filename}' not found — please choose it from the upload dialog "
          f"(Implementation/Datasets/kaggle/2/city_day.csv on your machine).")
    uploaded = files.upload()
    if filename not in uploaded and len(uploaded) == 1:
        # User uploaded under a different name — rename to expected filename.
        (only_name,) = uploaded.keys()
        Path(only_name).rename(filename)
    elif filename not in uploaded:
        raise FileNotFoundError(
            f"Upload did not contain '{filename}'. Uploaded: {list(uploaded.keys())}"
        )

    result = Path(filename)
    print(f"Uploaded dataset to: {result.resolve()}")
    return result


DATA_FILE = locate_or_upload_csv("city_day.csv")


Found dataset at: C:\Users\Sowmithra\Documents\literature-review\Implementation\Datasets\kaggle\2\city_day.csv


## 3. Configuration (transcribed from `config_phase1.py`)


In [6]:
# Feature and target columns (config_phase1.py)
FEATURE_COLUMNS = [
    "PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3",
    "Benzene", "Toluene", "Xylene"
]
TARGET_COLUMN = "AQI"

# Data split (chronological, not shuffled)
TRAIN_RATIO = 0.7
VAL_RATIO = 0.1
TEST_RATIO = 0.2

# Lookback periods for sequence-based models
LOOKBACK_PERIODS = [7, 14, 30]

# Model hyperparameters (config_phase1.py HYPERPARAMETERS dict).
# device is overridden below to the detected device (Known Fact #1) instead of the
# original hard-coded 'cpu'.
HYPERPARAMETERS = {
    "dlinear": {
        "seq_len": 14,              # overridden per lookback in the training loop
        "pred_len": 1,
        "seasonal_period": 7,       # weekly seasonality
        "epochs": 100,
        "batch_size": 32,
        "learning_rate": 0.001,
        "device": device,           # was 'cpu' in config_phase1.py; wired to detected device
    },
    "nlinear": {
        "seq_len": 14,
        "pred_len": 1,
        "epochs": 100,
        "batch_size": 32,
        "learning_rate": 0.001,
        "device": device,           # was 'cpu' in config_phase1.py; wired to detected device
    },
    "xgboost": {
        "n_estimators": 200,
        "max_depth": 7,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "random_state": RANDOM_SEED,
        "seq_len": 14,
        "pred_len": 1,
    },
    "lstm": {
        "seq_len": 14,
        "pred_len": 1,
    }
}

METRICS = ["rmse", "mae", "mape", "r2"]
VERBOSE = True

print(f"LOOKBACK_PERIODS = {LOOKBACK_PERIODS}")
print(f"FEATURE_COLUMNS ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}")
print(f"TARGET_COLUMN = {TARGET_COLUMN}")
print(f"Split ratios -> train={TRAIN_RATIO}, val={VAL_RATIO}, test={TEST_RATIO}")


LOOKBACK_PERIODS = [7, 14, 30]
FEATURE_COLUMNS (12): ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
TARGET_COLUMN = AQI
Split ratios -> train=0.7, val=0.1, test=0.2


## 4. Preprocessing (transcribed from `comparison/data_utils.py`)

Two data-preparation paths, exactly as in the source:

- **`AQIDataLoader`**: cleaning (per-city forward/backward fill, then mean-fill any
  remainder) + sliding-window sequence construction, used by DLinear, NLinear, and LSTM.
- **`XGBoostDataProcessor`**: lag features (`seq_len` lags per pollutant) + 7-day rolling
  mean/std, used by XGBoost.

Both split chronologically using `TRAIN_RATIO` / `VAL_RATIO` / `TEST_RATIO` — no shuffling.

---
📄 **Maps to manuscript:** Methods *Preprocessing*, *Train, validation and test split* (per-city 70/10/20 chronological), and *Baseline forecasting models*. This is the per-city evaluation protocol (chronological split, train-only scalers, strictly-lagged XGBoost features) used for every Table 2 baseline.

In [7]:
class AQIDataLoader:
    """Load and preprocess AQI data for baseline models (DLinear, NLinear, LSTM)."""

    def __init__(self, data_path=DATA_FILE, random_seed=RANDOM_SEED):
        self.data_path = Path(data_path)
        self.random_seed = random_seed
        np.random.seed(random_seed)
        if VERBOSE:
            print(f"[AQIDataLoader] Initialized with seed {random_seed}")

    def load_and_clean(self):
        """Load and clean data"""
        if VERBOSE:
            print(f"\n[AQIDataLoader] Loading data from {self.data_path}...")

        df = pd.read_csv(self.data_path)
        if VERBOSE:
            print(f"  Raw shape: {df.shape}")

        # Remove missing targets
        df = df.dropna(subset=[TARGET_COLUMN])

        # Handle missing values in features: forward-fill then backward-fill per city
        for col in FEATURE_COLUMNS:
            if col in df.columns:
                df[col] = df.groupby("City")[col].ffill().bfill()

        # Fill remaining NaNs with mean
        df = df.fillna(df[FEATURE_COLUMNS].mean())

        # Final cleanup
        df = df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN])

        if VERBOSE:
            print(f"  Cleaned shape: {df.shape}")

        return df

    def create_sequences(self, X, y, seq_len=14):
        """Create sliding window sequences"""
        X_seq, y_seq = [], []
        for i in range(len(X) - seq_len):
            X_seq.append(X[i:i + seq_len])
            y_seq.append(y[i + seq_len])
        return np.array(X_seq), np.array(y_seq)

    def prepare_data(self, df, seq_len=14):
        """Prepare data for DL models (DLinear, NLinear, LSTM).

        FIXED 2026-07-30 (Implementation/EVALUATION_PROTOCOL_AUDIT.md, L3/L4):
        sequences are windowed and split per city on chronological order, then
        concatenated, so the split is a genuine chronological forecast split
        rather than a city-block split disguised as one (L3). Scalers are fit
        on the training rows only (L4). Numbers produced before this fix are
        STALE and not comparable to anything computed after it.
        """
        if VERBOSE:
            print(f"\n[AQIDataLoader] Preparing sequences (seq_len={seq_len})...")

        cities = sorted(df["City"].unique())
        city_raw = {}
        for city in cities:
            df_c = df[df["City"] == city].sort_values("Date").reset_index(drop=True)
            if len(df_c) < seq_len + 10:
                continue
            city_raw[city] = {
                "X": df_c[FEATURE_COLUMNS].values.astype(np.float64),
                "y": df_c[TARGET_COLUMN].values.astype(np.float64),
                "n": len(df_c),
            }
        valid_cities = list(city_raw.keys())

        # Per-city split indices, computed on the number of sequences that
        # city can produce (n_rows - seq_len).
        city_splits = {}
        for city in valid_cities:
            n = city_raw[city]["n"]
            n_seq = n - seq_len
            train_end = int(n_seq * TRAIN_RATIO)
            val_end = train_end + int(n_seq * VAL_RATIO)
            city_splits[city] = (train_end, val_end, n_seq)

        # Scalers fit on the training rows only (first TRAIN_RATIO*n raw rows
        # of each city, matching the fraction used for training sequences).
        train_X_list, train_y_list = [], []
        for city in valid_cities:
            n = city_raw[city]["n"]
            n_train_rows = int(n * TRAIN_RATIO)
            train_X_list.append(city_raw[city]["X"][:n_train_rows])
            train_y_list.append(city_raw[city]["y"][:n_train_rows])

        scaler_X = MinMaxScaler(feature_range=(0, 1)).fit(np.vstack(train_X_list))
        scaler_y = MinMaxScaler(feature_range=(0, 1)).fit(
            np.concatenate(train_y_list).reshape(-1, 1)
        )

        X_train, y_train = [], []
        X_val, y_val = [], []
        X_test, y_test = [], []

        for city in valid_cities:
            d = city_raw[city]
            train_end, val_end, n_seq = city_splits[city]

            X_sc = scaler_X.transform(d["X"])
            y_sc = scaler_y.transform(d["y"].reshape(-1, 1)).ravel()

            for i in range(n_seq):
                X_win = X_sc[i:i + seq_len]
                y_tgt = y_sc[i + seq_len]
                if i < train_end:
                    X_train.append(X_win)
                    y_train.append(y_tgt)
                elif i < val_end:
                    X_val.append(X_win)
                    y_val.append(y_tgt)
                else:
                    X_test.append(X_win)
                    y_test.append(y_tgt)

        X_train = np.array(X_train)
        y_train = np.array(y_train).reshape(-1, 1)
        X_val = np.array(X_val)
        y_val = np.array(y_val).reshape(-1, 1)
        X_test = np.array(X_test)
        y_test = np.array(y_test).reshape(-1, 1)

        if VERBOSE:
            print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

        return {
            "X_train": X_train, "y_train": y_train,
            "X_val": X_val, "y_val": y_val,
            "X_test": X_test, "y_test": y_test,
            "scaler_X": scaler_X,
            "scaler_y": scaler_y,
            "feature_names": FEATURE_COLUMNS,
            "valid_cities": valid_cities,
        }


class XGBoostDataProcessor:
    """Prepare data for XGBoost (uses lag features instead of sequences).

    FIXED 2026-07-30 (see Implementation/EVALUATION_PROTOCOL_AUDIT.md, L1/L2/L3/L4):
    lag/rolling features are now computed per city (fixes L2 cross-city bleed);
    same-day pollutant columns never enter feature_cols (fixes L1 contemporaneous
    target leakage); the split is computed per city on chronological order then
    concatenated (fixes L3 city-block split); scaler_y is fit on the training
    split only (fixes L4 pre-split scaler). Numbers produced before this fix are
    STALE and not comparable to anything computed after it.
    """

    def __init__(self, seq_len=14, random_seed=RANDOM_SEED):
        self.seq_len = seq_len
        self.random_seed = random_seed
        np.random.seed(random_seed)

    def create_lag_features(self, df):
        """Create per-city lag and rolling features for tree-based models"""
        if VERBOSE:
            print(f"\n[XGBoostDataProcessor] Creating lag features (seq_len={self.seq_len})...")

        df_copy = df.sort_values(["City", "Date"]).reset_index(drop=True).copy()
        grouped = df_copy.groupby("City")

        # Lag features: strictly past values, grouped per city so no city's
        # trailing rows bleed into the next city's leading rows.
        for col in FEATURE_COLUMNS:
            for lag in range(1, self.seq_len + 1):
                df_copy[f"{col}_lag_{lag}"] = grouped[col].shift(lag)

        # Rolling statistics computed over the lag-1 series (days [t-7, t-1]),
        # never the same-day column, so they cannot see day t itself.
        for col in FEATURE_COLUMNS:
            shifted = grouped[col].shift(1)
            df_copy[f"{col}_rolling_mean_7"] = shifted.groupby(df_copy["City"]).transform(
                lambda s: s.rolling(7).mean()
            )
            df_copy[f"{col}_rolling_std_7"] = shifted.groupby(df_copy["City"]).transform(
                lambda s: s.rolling(7).std()
            )

        # Remove NaN rows created by lagging (the first several rows of each city)
        df_copy = df_copy.dropna().reset_index(drop=True)

        if VERBOSE:
            print(f"  Features created: {df_copy.shape[1]}")

        return df_copy

    def prepare_data(self, df):
        """Prepare data for XGBoost"""
        df_processed = self.create_lag_features(df)

        # Only engineered lag/rolling columns are features. Same-day pollutant
        # values (which mathematically determine same-day AQI via the CPCB
        # sub-index formula) are excluded entirely, along with City/Date/AQI/AQI_Bucket.
        feature_cols = [col for col in df_processed.columns
                         if col not in FEATURE_COLUMNS
                         and col not in [TARGET_COLUMN, "AQI_Bucket", "City", "Date"]]

        # Per-city chronological split, then concatenate across cities.
        train_frames, val_frames, test_frames = [], [], []
        for _, g in df_processed.groupby("City"):
            g = g.sort_values("Date").reset_index(drop=True)
            n = len(g)
            train_idx = int(n * TRAIN_RATIO)
            val_idx = train_idx + int(n * VAL_RATIO)
            train_frames.append(g.iloc[:train_idx])
            val_frames.append(g.iloc[train_idx:val_idx])
            test_frames.append(g.iloc[val_idx:])

        df_train = pd.concat(train_frames, ignore_index=True)
        df_val = pd.concat(val_frames, ignore_index=True)
        df_test = pd.concat(test_frames, ignore_index=True)

        # Scaler fit on the training split only.
        scaler_y = MinMaxScaler()
        scaler_y.fit(df_train[TARGET_COLUMN].values.reshape(-1, 1))

        X_train = df_train[feature_cols].values
        y_train = scaler_y.transform(df_train[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

        X_val = df_val[feature_cols].values
        y_val = scaler_y.transform(df_val[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

        X_test = df_test[feature_cols].values
        y_test = scaler_y.transform(df_test[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

        if VERBOSE:
            print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

        return {
            "X_train": X_train, "y_train": y_train,
            "X_val": X_val, "y_val": y_val,
            "X_test": X_test, "y_test": y_test,
            "scaler_y": scaler_y,
            "feature_names": feature_cols
        }


### 4.1 Evaluation metrics (transcribed from `comparison/evaluate.py`)


In [8]:
class Evaluator:
    """Compute evaluation metrics for AQI predictions"""

    @staticmethod
    def rmse(y_true, y_pred):
        return np.sqrt(mean_squared_error(y_true, y_pred))

    @staticmethod
    def mae(y_true, y_pred):
        return mean_absolute_error(y_true, y_pred)

    @staticmethod
    def mape(y_true, y_pred):
        y_true = y_true.flatten()
        y_pred = y_pred.flatten()
        mask = y_true != 0
        if mask.sum() == 0:
            return 0.0
        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    @staticmethod
    def r2(y_true, y_pred):
        return r2_score(y_true.flatten(), y_pred.flatten())

    @staticmethod
    def evaluate(y_true, y_pred, model_name="Model"):
        y_true = y_true.flatten()
        y_pred = y_pred.flatten()

        metrics = {
            "model": model_name,
            "rmse": Evaluator.rmse(y_true, y_pred),
            "mae": Evaluator.mae(y_true, y_pred),
            "mape": Evaluator.mape(y_true, y_pred),
            "r2": Evaluator.r2(y_true, y_pred),
        }

        if VERBOSE:
            print(f"\n[{model_name}] Evaluation Results:")
            print(f"  RMSE:  {metrics['rmse']:.6f}")
            print(f"  MAE:   {metrics['mae']:.6f}")
            print(f"  MAPE:  {metrics['mape']:.4f}%")
            print(f"  R\u00b2:    {metrics['r2']:.6f}")

        return metrics


class ComparisonReport:
    """Generate comparison reports across models"""

    def __init__(self):
        self.results = {}

    def add_result(self, model_name, y_true, y_pred):
        metrics = Evaluator.evaluate(y_true, y_pred, model_name)
        self.results[model_name] = metrics
        return metrics

    def get_summary(self):
        if not self.results:
            return None
        return {
            "rmse": {k: v["rmse"] for k, v in self.results.items()},
            "mae": {k: v["mae"] for k, v in self.results.items()},
            "mape": {k: v["mape"] for k, v in self.results.items()},
            "r2": {k: v["r2"] for k, v in self.results.items()},
        }

    def print_summary(self):
        if not self.results:
            print("No results to display")
            return

        print("\n" + "=" * 80)
        print("PHASE 1 BASELINE COMPARISON SUMMARY")
        print("=" * 80)
        print(f"\n{'Model':<20} {'RMSE':<15} {'MAE':<15} {'MAPE':<15} {'R\u00b2':<15}")
        print("-" * 80)

        sorted_results = sorted(self.results.items(), key=lambda x: x[1]["r2"], reverse=True)
        for model_name, metrics in sorted_results:
            print(f"{model_name:<20} {metrics['rmse']:<15.6f} {metrics['mae']:<15.6f} "
                  f"{metrics['mape']:<15.4f}% {metrics['r2']:<15.6f}")
        print("=" * 80)


print("Evaluator and ComparisonReport defined.")


Evaluator and ComparisonReport defined.


### 4.2 Load and clean the dataset once


In [9]:
loader = AQIDataLoader(data_path=DATA_FILE, random_seed=RANDOM_SEED)
df = loader.load_and_clean()
df.head()


[AQIDataLoader] Initialized with seed 42

[AQIDataLoader] Loading data from ..\Datasets\kaggle\2\city_day.csv...
  Raw shape: (29531, 16)
  Cleaned shape: (24850, 16)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
28,Ahmedabad,2015-01-29,83.13,122.41,6.93,28.71,33.72,25.63,6.93,49.52,59.76,0.02,0.00,3.14,209.0,Poor
29,Ahmedabad,2015-01-30,79.84,122.41,13.85,28.68,41.08,25.63,13.85,48.49,97.07,0.04,0.00,4.81,328.0,Very Poor
30,Ahmedabad,2015-01-31,94.52,122.41,24.39,32.66,52.61,25.63,24.39,67.39,111.33,0.24,0.01,7.67,514.0,Severe
31,Ahmedabad,2015-02-01,135.99,122.41,43.48,42.08,84.57,25.63,43.48,75.23,102.70,0.40,0.04,25.87,782.0,Severe
32,Ahmedabad,2015-02-02,178.33,122.41,54.56,35.31,72.80,25.63,54.56,55.04,107.38,0.46,0.06,35.61,914.0,Severe


## 5. Model 1 — DLinear

Decomposition-Linear model (Zeng et al., "Are Transformers Effective for Time Series
Forecasting?", AAAI 2023, https://arxiv.org/pdf/2205.13504). Decomposes the input window
into trend (moving average) + seasonal (residual), applies a separate linear layer to
each, and sums the results. Transcribed from `models/dlinear.py`; the only change is
that `device` is passed in as the detected GPU/CPU device instead of the hard-coded
`'cpu'` default.

---
📄 **Maps to manuscript:** **Table 2 (`tab:main_results`)** DLinear row (R2 = 0.795 @ LB7) and Methods *DLinear and NLinear*. Uses a scalar output head.

In [10]:
class DecompositionBlock(nn.Module):
    """Decompose series into trend and seasonal"""

    def __init__(self, kernel_size=25):
        super(DecompositionBlock, self).__init__()
        self.kernel_size = kernel_size
        self.moving_avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        """x: (batch, seq_len, num_features)"""
        batch_size, seq_len, num_features = x.size()
        padding = (self.kernel_size - 1) // 2

        x_padded = nn.functional.pad(x.permute(0, 2, 1), (padding, padding), mode="reflect")
        trend = self.moving_avg(x_padded)
        trend = trend.permute(0, 2, 1)
        seasonal = x - trend

        return trend, seasonal


class DLinear(nn.Module):
    """DLinear Model: Decomposition-Linear"""

    def __init__(self, seq_len, pred_len, num_features, seasonal_period=7):
        super(DLinear, self).__init__()

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.num_features = num_features

        self.decomposition = DecompositionBlock(kernel_size=seasonal_period)
        self.trend_linear = nn.Linear(seq_len, pred_len)
        self.seasonal_linear = nn.Linear(seq_len, pred_len)
        # FIXED 2026-07-30: scalar output head. Previously all num_features
        # output channels were trained toward the same broadcast AQI target
        # and channel 0 was read back at test time (see EVALUATION_PROTOCOL_
        # AUDIT.md, "channel-0 output bug"). This projects the per-channel
        # trend+seasonal reconstruction down to a single AQI estimate.
        self.output_head = nn.Linear(num_features, 1)

    def forward(self, x):
        """x: (batch, seq_len, num_features) -> (batch, pred_len, 1)"""
        trend, seasonal = self.decomposition(x)

        trend_pred = self.trend_linear(trend.permute(0, 2, 1)).permute(0, 2, 1)
        seasonal_pred = self.seasonal_linear(seasonal.permute(0, 2, 1)).permute(0, 2, 1)

        output = trend_pred + seasonal_pred      # (batch, pred_len, num_features)
        output = self.output_head(output)        # (batch, pred_len, 1)
        return output


class DLinearTrainer:
    """Trainer for DLinear model"""

    def __init__(self, seq_len, pred_len, num_features, seasonal_period=7,
                 epochs=100, batch_size=32, learning_rate=0.001, device="cpu"):

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.num_features = num_features
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.device = device

        self.model = DLinear(seq_len, pred_len, num_features, seasonal_period).to(device)

        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)

        self.train_loss_history = []
        self.val_loss_history = []

        if VERBOSE:
            print(f"[DLinear] Model initialized on {device}")
            print(f"  Input: (batch, {seq_len}, {num_features})")
            print(f"  Output: (batch, {pred_len}, 1)")

    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(self.device)
            y_batch = y_batch.to(self.device)

            output = self.model(X_batch)

            # Target is (batch, pred_len); expand to match (batch, pred_len, num_features)
            y_batch_expanded = y_batch.unsqueeze(-1).expand_as(output)
            loss = self.criterion(output, y_batch_expanded)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()

        return total_loss / len(train_loader)

    def validate(self, val_loader):
        self.model.eval()
        total_loss = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                output = self.model(X_batch)
                y_batch_expanded = y_batch.unsqueeze(-1).expand_as(output)
                loss = self.criterion(output, y_batch_expanded)
                total_loss += loss.item()

        return total_loss / len(val_loader)

    def fit(self, X_train, y_train, X_val, y_val):
        if VERBOSE:
            print(f"\n[DLinear] Training for {self.epochs} epochs...")

        train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=False)

        val_dataset = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val))
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False)

        best_val_loss = float("inf")
        patience = 10
        patience_counter = 0

        for epoch in range(self.epochs):
            train_loss = self.train_epoch(train_loader)
            val_loss = self.validate(val_loader)

            self.train_loss_history.append(train_loss)
            self.val_loss_history.append(val_loss)

            if VERBOSE and (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{self.epochs}: Train Loss={train_loss:.6f}, Val Loss={val_loss:.6f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    if VERBOSE:
                        print(f"  Early stopping at epoch {epoch+1}")
                    break

        if VERBOSE:
            print(f"[DLinear] Training complete. Best val loss: {best_val_loss:.6f}")

    def predict(self, X_test):
        self.model.eval()
        with torch.no_grad():
            X_test_tensor = torch.FloatTensor(X_test).to(self.device)
            predictions = self.model(X_test_tensor).cpu().numpy()
        return predictions


print("DLinear architecture + trainer defined.")


DLinear architecture + trainer defined.


## 6. Model 2 — NLinear

Normalization-Linear model (same paper as DLinear). Subtracts the last timestep's value
from the window (handles distribution shift), applies a single linear layer, then adds
the last value back. Transcribed from `models/nlinear.py`, same `device` wiring change
as DLinear.

---
📄 **Maps to manuscript:** **Table 2 (`tab:main_results`)** NLinear row (R2 = 0.807 @ LB7) and Methods *DLinear and NLinear*.

In [11]:
class NLinear(nn.Module):
    """NLinear Model: Normalization-Linear"""

    def __init__(self, seq_len, pred_len, num_features):
        super(NLinear, self).__init__()

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.num_features = num_features

        self.linear = nn.Linear(seq_len, pred_len)
        # FIXED 2026-07-30: scalar output head, same rationale as DLinear
        # above (see EVALUATION_PROTOCOL_AUDIT.md, "channel-0 output bug").
        self.output_head = nn.Linear(num_features, 1)

    def forward(self, x):
        """x: (batch, seq_len, num_features) -> (batch, pred_len, 1)"""
        batch_size, seq_len, num_features = x.size()

        last_value = x[:, -1:, :]
        x_normalized = x - last_value

        x_perm = x_normalized.permute(0, 2, 1)
        output_norm = self.linear(x_perm)
        output_norm = output_norm.permute(0, 2, 1)

        output = output_norm + last_value       # (batch, pred_len, num_features)
        output = self.output_head(output)        # (batch, pred_len, 1)
        return output


class NLinearTrainer:
    """Trainer for NLinear model"""

    def __init__(self, seq_len, pred_len, num_features,
                 epochs=100, batch_size=32, learning_rate=0.001, device="cpu"):

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.num_features = num_features
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.device = device

        self.model = NLinear(seq_len, pred_len, num_features).to(device)

        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)

        self.train_loss_history = []
        self.val_loss_history = []

        if VERBOSE:
            print(f"[NLinear] Model initialized on {device}")
            print(f"  Input: (batch, {seq_len}, {num_features})")
            print(f"  Output: (batch, {pred_len}, 1)")

    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(self.device)
            y_batch = y_batch.to(self.device)

            output = self.model(X_batch)
            y_batch_expanded = y_batch.unsqueeze(-1).expand_as(output)
            loss = self.criterion(output, y_batch_expanded)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()

        return total_loss / len(train_loader)

    def validate(self, val_loader):
        self.model.eval()
        total_loss = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                output = self.model(X_batch)
                y_batch_expanded = y_batch.unsqueeze(-1).expand_as(output)
                loss = self.criterion(output, y_batch_expanded)
                total_loss += loss.item()

        return total_loss / len(val_loader)

    def fit(self, X_train, y_train, X_val, y_val):
        if VERBOSE:
            print(f"\n[NLinear] Training for {self.epochs} epochs...")

        train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=False)

        val_dataset = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val))
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False)

        best_val_loss = float("inf")
        patience = 10
        patience_counter = 0

        for epoch in range(self.epochs):
            train_loss = self.train_epoch(train_loader)
            val_loss = self.validate(val_loader)

            self.train_loss_history.append(train_loss)
            self.val_loss_history.append(val_loss)

            if VERBOSE and (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{self.epochs}: Train Loss={train_loss:.6f}, Val Loss={val_loss:.6f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    if VERBOSE:
                        print(f"  Early stopping at epoch {epoch+1}")
                    break

        if VERBOSE:
            print(f"[NLinear] Training complete. Best val loss: {best_val_loss:.6f}")

    def predict(self, X_test):
        self.model.eval()
        with torch.no_grad():
            X_test_tensor = torch.FloatTensor(X_test).to(self.device)
            predictions = self.model(X_test_tensor).cpu().numpy()
        return predictions


print("NLinear architecture + trainer defined.")


NLinear architecture + trainer defined.


## 7. Model 3 — XGBoost

Gradient-boosted trees over lag features (Chen & Guestrin, "XGBoost: A Scalable Tree
Boosting System", KDD 2016). Transcribed from `models/xgboost_model.py`. `random_state`
already defaults to 42 in the source and is read from `HYPERPARAMETERS['xgboost']`
here — now explicitly tied to the same `RANDOM_SEED` used everywhere else in the notebook.
**Correction**: `tree_method='hist'` does *not* use the GPU automatically in modern
XGBoost (>=2.0) — GPU execution requires an explicit `device='cuda'` parameter, which
`xgboost_model.py`'s source config never sets. This model therefore trains on CPU here,
same as the original pipeline (consistent with the equivalent note in the
`phase2_ensemble` Colab notebook's XGBoost section). `random_state` still makes it
fully reproducible either way.

---
📄 **Maps to manuscript:** **Table 2 (`tab:main_results`)** XGBoost row (R2 = 0.864 @ LB7 -- best on every metric) and Methods *Extreme Gradient Boosting (XGBoost)*. Features are strictly-lagged (no same-day pollutants).

In [12]:
class XGBoostModel:
    """XGBoost wrapper for AQI prediction"""

    def __init__(self, **kwargs):
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "tree_method": "hist",
            "random_state": RANDOM_SEED,
        }
        params.update(kwargs)

        self.model = xgb.XGBRegressor(**params)
        self.feature_names = None
        self.train_history = {}

        if VERBOSE:
            print(f"[XGBoost] Model initialized")
            print(f"  n_estimators: {self.model.n_estimators}")
            print(f"  max_depth: {self.model.max_depth}")
            print(f"  learning_rate: {self.model.learning_rate}")

    def fit(self, X_train, y_train, X_val, y_val, feature_names=None):
        self.feature_names = feature_names

        if VERBOSE:
            print(f"\n[XGBoost] Training with {X_train.shape[0]} samples, {X_train.shape[1]} features...")

        self.model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        if VERBOSE:
            print(f"[XGBoost] Training complete with {self.model.n_estimators} estimators")

    def predict(self, X_test):
        predictions = self.model.predict(X_test)
        return predictions.reshape(-1, 1)

    def get_feature_importance(self, top_n=20):
        if self.feature_names is None:
            feature_names = [f"feature_{i}" for i in range(len(self.model.feature_importances_))]
        else:
            feature_names = self.feature_names

        importances = self.model.get_booster().get_score(importance_type="weight")
        sorted_features = sorted(importances.items(), key=lambda x: x[1], reverse=True)[:top_n]
        return dict(sorted_features)


def create_xgboost_model(config_params=None):
    if config_params is None:
        config_params = HYPERPARAMETERS["xgboost"]

    xgb_params = {
        "n_estimators": config_params.get("n_estimators", 200),
        "max_depth": config_params.get("max_depth", 7),
        "learning_rate": config_params.get("learning_rate", 0.1),
        "subsample": config_params.get("subsample", 0.8),
        "colsample_bytree": config_params.get("colsample_bytree", 0.8),
        "reg_alpha": config_params.get("reg_alpha", 0.1),
        "reg_lambda": config_params.get("reg_lambda", 1.0),
        "random_state": config_params.get("random_state", RANDOM_SEED),
    }

    return XGBoostModel(**xgb_params)


print("XGBoostModel + create_xgboost_model defined.")


XGBoostModel + create_xgboost_model defined.


## 8. Model 4 — LSTM (Keras/TensorFlow baseline)

Stacked LSTM regression baseline (64 -> 32 units, with dropout), transcribed from
`models/lstm_wrapper.py`. This is trained/compared alongside DLinear, NLinear, and
XGBoost in `compare_lookback_comprehensive.py` at all three lookback periods — that
script is the one confirming LSTM is part of this phase's roster (not just
`run_comparison.py`, which only covers DLinear/NLinear/XGBoost). `device` is accepted
for interface parity with the PyTorch trainers but ignored — TensorFlow places ops on
the GPU automatically when one is visible (confirmed by the `tf.config.list_physical_devices`
print above).

---
📄 **Maps to manuscript:** This notebook's own conventional LSTM baseline (R2 = 0.813 @ LB7). Note: the manuscript's **Table 2** LSTM row (0.818) is the PyTorch LSTM trained in `phase4_advanced_v2_colab_4.ipynb` Section 7c under the same pipeline; the two land in the same range.

In [13]:
class LSTMTrainer:
    """LSTM trainer matching the interface of DLinearTrainer / NLinearTrainer"""

    def __init__(self, lookback=None, seq_len=None, num_features=None, pred_len=1, epochs=100,
                 batch_size=32, learning_rate=0.001, device="cpu", num_classes=6):
        if lookback is not None:
            self.seq_len = lookback
        elif seq_len is not None:
            self.seq_len = seq_len
        else:
            raise ValueError("Must provide either 'lookback' or 'seq_len' parameter")

        if num_features is None:
            raise ValueError("Must provide 'num_features' parameter")
        self.num_features = num_features
        self.pred_len = pred_len
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.num_classes = num_classes
        self.model = None
        self.history = None

        if VERBOSE:
            print(f"[LSTM] Initialized with seq_len={self.seq_len}, num_features={num_features}")

    def build_model(self):
        """Build LSTM regression model"""
        model = Sequential([
            LSTM(64, activation="relu", return_sequences=True,
                 input_shape=(self.seq_len, self.num_features)),
            Dropout(0.2),
            LSTM(32, activation="relu", return_sequences=False),
            Dropout(0.2),
            Dense(16, activation="relu"),
            Dropout(0.1),
            Dense(1)
        ])

        model.compile(
            optimizer=Adam(learning_rate=self.learning_rate),
            loss="mse",
            metrics=["mae"]
        )

        if VERBOSE:
            print(f"[LSTM] Model built successfully")

        return model

    def fit(self, X_train, y_train, X_val, y_val, verbose=0):
        if VERBOSE:
            print(f"\n[LSTM] Training for {self.epochs} epochs...")

        self.model = self.build_model()

        X_train = np.asarray(X_train, dtype=np.float32)
        y_train = np.asarray(y_train, dtype=np.float32).reshape(-1, 1)
        X_val = np.asarray(X_val, dtype=np.float32)
        y_val = np.asarray(y_val, dtype=np.float32).reshape(-1, 1)

        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )

        self.history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=self.epochs,
            batch_size=self.batch_size,
            callbacks=[early_stopping],
            verbose=0
        )

        best_epoch = np.argmin(self.history.history["val_loss"])
        if VERBOSE:
            print(f"[LSTM] Training complete. Best epoch: {best_epoch + 1}")
            print(f"  Best val_loss: {self.history.history['val_loss'][best_epoch]:.6f}")

    def predict(self, X_test):
        X_test = np.asarray(X_test, dtype=np.float32)
        predictions = self.model.predict(X_test, verbose=0)
        return predictions


print("LSTMTrainer defined.")


LSTMTrainer defined.


## 9. Train + evaluate all four models across all lookback periods

Mirrors `compare_lookback_comprehensive.py`: for each lookback in `LOOKBACK_PERIODS`
(7, 14, 30 days), prepare both data representations (sequences for DLinear/NLinear/LSTM,
lag features for XGBoost), train all four models, and evaluate on the held-out test split.
`y_test_actual` (the inverse-scaled ground truth) is computed once, immediately after
`dl_data` is prepared, so it is available to every model.

**Evaluation protocol.** The pipeline uses a per-city, strictly causal protocol: a per-city
chronological 70/10/20 train/validation/test split, XGBoost features built from strictly
lagged pollutant values (no same-day pollutants) with per-city lag/rolling windows, scalers
fit on the training split only, and a scalar output head for DLinear/NLinear. The numbers
printed below and saved to the comparison CSV/JSON are the baseline values reported in the
manuscript's Table 2.

---
📄 **Maps to manuscript:** **Table 2 (`tab:main_results`)** baseline column -- DLinear / NLinear / XGBoost rows at LB = 7 are the manuscript values; LB 14/30 are reported here for completeness.

In [14]:
all_results = {}     # {lookback: {model_name: metrics_dict}}
result_rows = []      # flat rows for the final DataFrame

for lookback in LOOKBACK_PERIODS:
    print(f"\n{'=' * 100}")
    print(f"LOOKBACK PERIOD: {lookback} days")
    print(f"{'=' * 100}")

    results_key = f"lookback_{lookback}"
    all_results[results_key] = {}

    # ---- Prepare data -------------------------------------------------
    dl_data = loader.prepare_data(df, seq_len=lookback)
    y_test_actual = dl_data["scaler_y"].inverse_transform(dl_data["y_test"].reshape(-1, 1))

    xgb_processor = XGBoostDataProcessor(seq_len=lookback, random_seed=RANDOM_SEED)
    xgb_data = xgb_processor.prepare_data(df)
    y_test_actual_xgb = xgb_data["scaler_y"].inverse_transform(xgb_data["y_test"].reshape(-1, 1))

    report = ComparisonReport()

    # ---- DLinear --------------------------------------------------------
    print(f"\n[DLinear] lookback={lookback}...")
    try:
        cfg = HYPERPARAMETERS["dlinear"].copy()
        trainer_dl = DLinearTrainer(
            seq_len=lookback,
            pred_len=cfg["pred_len"],
            num_features=len(dl_data["feature_names"]),
            seasonal_period=cfg["seasonal_period"],
            epochs=cfg["epochs"],
            batch_size=cfg["batch_size"],
            learning_rate=cfg["learning_rate"],
            device=cfg["device"],
        )
        trainer_dl.fit(dl_data["X_train"], dl_data["y_train"], dl_data["X_val"], dl_data["y_val"])

        y_pred_dlinear = trainer_dl.predict(dl_data["X_test"])
        y_pred_dlinear = y_pred_dlinear[:, 0, 0].reshape(-1, 1)
        y_pred_dlinear = dl_data["scaler_y"].inverse_transform(y_pred_dlinear)

        metrics_dl = Evaluator.evaluate(y_test_actual, y_pred_dlinear, "DLinear")
        report.add_result("DLinear", y_test_actual, y_pred_dlinear)
        all_results[results_key]["DLinear"] = metrics_dl
        result_rows.append({"model": "DLinear", "lookback": lookback, **{k: metrics_dl[k] for k in METRICS}})
        print(f"  [OK] DLinear - RMSE={metrics_dl['rmse']:.2f}, R2={metrics_dl['r2']:.4f}")
    except Exception as e:
        print(f"  [FAIL] DLinear failed: {e}")
        all_results[results_key]["DLinear"] = None

    # ---- NLinear --------------------------------------------------------
    print(f"\n[NLinear] lookback={lookback}...")
    try:
        cfg = HYPERPARAMETERS["nlinear"].copy()
        trainer_nl = NLinearTrainer(
            seq_len=lookback,
            pred_len=cfg["pred_len"],
            num_features=len(dl_data["feature_names"]),
            epochs=cfg["epochs"],
            batch_size=cfg["batch_size"],
            learning_rate=cfg["learning_rate"],
            device=cfg["device"],
        )
        trainer_nl.fit(dl_data["X_train"], dl_data["y_train"], dl_data["X_val"], dl_data["y_val"])

        y_pred_nlinear = trainer_nl.predict(dl_data["X_test"])
        y_pred_nlinear = y_pred_nlinear[:, 0, 0].reshape(-1, 1)
        y_pred_nlinear = dl_data["scaler_y"].inverse_transform(y_pred_nlinear)

        metrics_nl = Evaluator.evaluate(y_test_actual, y_pred_nlinear, "NLinear")
        report.add_result("NLinear", y_test_actual, y_pred_nlinear)
        all_results[results_key]["NLinear"] = metrics_nl
        result_rows.append({"model": "NLinear", "lookback": lookback, **{k: metrics_nl[k] for k in METRICS}})
        print(f"  [OK] NLinear - RMSE={metrics_nl['rmse']:.2f}, R2={metrics_nl['r2']:.4f}")
    except Exception as e:
        print(f"  [FAIL] NLinear failed: {e}")
        all_results[results_key]["NLinear"] = None

    # ---- XGBoost ----------------------------------------------------------
    print(f"\n[XGBoost] lookback={lookback}...")
    try:
        xgb_config = HYPERPARAMETERS["xgboost"].copy()
        xgb_config["seq_len"] = lookback
        xgb_model = create_xgboost_model(xgb_config)

        xgb_model.fit(
            xgb_data["X_train"], xgb_data["y_train"],
            xgb_data["X_val"], xgb_data["y_val"],
            feature_names=xgb_data["feature_names"],
        )

        y_pred_xgb = xgb_model.predict(xgb_data["X_test"])
        y_pred_xgb = xgb_data["scaler_y"].inverse_transform(y_pred_xgb)

        metrics_xgb = Evaluator.evaluate(y_test_actual_xgb, y_pred_xgb, "XGBoost")
        report.add_result("XGBoost", y_test_actual_xgb, y_pred_xgb)
        all_results[results_key]["XGBoost"] = metrics_xgb
        result_rows.append({"model": "XGBoost", "lookback": lookback, **{k: metrics_xgb[k] for k in METRICS}})
        print(f"  [OK] XGBoost - RMSE={metrics_xgb['rmse']:.2f}, R2={metrics_xgb['r2']:.4f}")
    except Exception as e:
        print(f"  [FAIL] XGBoost failed: {e}")
        all_results[results_key]["XGBoost"] = None

    # ---- LSTM ---------------------------------------------------------
    print(f"\n[LSTM] lookback={lookback}...")
    try:
        lstm_trainer = LSTMTrainer(
            lookback=lookback,
            num_features=len(FEATURE_COLUMNS),
            epochs=HYPERPARAMETERS["lstm"].get("epochs", 100),
            batch_size=HYPERPARAMETERS["lstm"].get("batch_size", 32),
        )
        lstm_trainer.fit(dl_data["X_train"], dl_data["y_train"], dl_data["X_val"], dl_data["y_val"], verbose=0)

        y_pred_lstm = lstm_trainer.predict(dl_data["X_test"])
        y_pred_lstm = dl_data["scaler_y"].inverse_transform(y_pred_lstm)

        metrics_lstm = Evaluator.evaluate(y_test_actual, y_pred_lstm, "LSTM")
        report.add_result("LSTM", y_test_actual, y_pred_lstm)
        all_results[results_key]["LSTM"] = metrics_lstm
        result_rows.append({"model": "LSTM", "lookback": lookback, **{k: metrics_lstm[k] for k in METRICS}})
        print(f"  [OK] LSTM - RMSE={metrics_lstm['rmse']:.2f}, R2={metrics_lstm['r2']:.4f}")
    except Exception as e:
        print(f"  [FAIL] LSTM failed: {e}")
        all_results[results_key]["LSTM"] = None

    report.print_summary()

print("\n\nAll lookback periods complete.")



LOOKBACK PERIOD: 7 days

[AQIDataLoader] Preparing sequences (seq_len=7)...
  Train: (17253, 7, 12), Val: (2454, 7, 12), Test: (4961, 7, 12)

[XGBoostDataProcessor] Creating lag features (seq_len=7)...


C:\Users\Sowmithra\AppData\Local\Temp\ipykernel_7680\2248138011.py:180: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f"{col}_rolling_std_7"] = shifted.groupby(df_copy["City"]).transform(
C:\Users\Sowmithra\AppData\Local\Temp\ipykernel_7680\2248138011.py:177: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f"{col}_rolling_mean_7"] = shifted.groupby(df_copy["City"]).transform(


  Features created: 124
  Train: (17253, 108), Val: (2454, 108), Test: (4961, 108)

[DLinear] lookback=7...
[DLinear] Model initialized on cpu
  Input: (batch, 7, 12)
  Output: (batch, 1, 1)

[DLinear] Training for 100 epochs...
  Epoch 10/100: Train Loss=0.001869, Val Loss=0.000728
  Epoch 20/100: Train Loss=0.001434, Val Loss=0.000548
  Epoch 30/100: Train Loss=0.001431, Val Loss=0.000545
  Epoch 40/100: Train Loss=0.001432, Val Loss=0.000542
  Epoch 50/100: Train Loss=0.001432, Val Loss=0.000539
  Epoch 60/100: Train Loss=0.001433, Val Loss=0.000537
  Epoch 70/100: Train Loss=0.001433, Val Loss=0.000535
  Epoch 80/100: Train Loss=0.001433, Val Loss=0.000534
  Epoch 90/100: Train Loss=0.001434, Val Loss=0.000533
  Epoch 100/100: Train Loss=0.001434, Val Loss=0.000533
[DLinear] Training complete. Best val loss: 0.000533

[DLinear] Evaluation Results:
  RMSE:  50.634005
  MAE:   26.105681
  MAPE:  19.7487%
  R²:    0.795119

[DLinear] Evaluation Results:
  RMSE:  50.634005
  MAE:   26.

c:\Users\Sowmithra\Documents\literature-review\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[LSTM] Model built successfully
[LSTM] Training complete. Best epoch: 9
  Best val_loss: 0.000513

[LSTM] Evaluation Results:
  RMSE:  48.338352
  MAE:   29.935002
  MAPE:  31.2232%
  R²:    0.813275

[LSTM] Evaluation Results:
  RMSE:  48.338352
  MAE:   29.935002
  MAPE:  31.2232%
  R²:    0.813275
  [OK] LSTM - RMSE=48.34, R2=0.8133

PHASE 1 BASELINE COMPARISON SUMMARY

Model                RMSE            MAE             MAPE            R²             
--------------------------------------------------------------------------------
XGBoost              41.291148       21.768433       18.3163        % 0.863752       
LSTM                 48.338352       29.935002       31.2232        % 0.813275       
NLinear              49.203092       25.683065       18.7882        % 0.806535       
DLinear              50.634005       26.105681       19.7487        % 0.795119       

LOOKBACK PERIOD: 14 days

[AQIDataLoader] Preparing sequences (seq_len=14)...
  Train: (17126, 14, 12), Val: (243

...

[LSTM] Training complete. Best epoch: 10
  Best val_loss: 0.000497

[LSTM] Evaluation Results:
  RMSE:  48.480389
  MAE:   27.710174
  MAPE:  26.2486%
  R²:    0.812660

[LSTM] Evaluation Results:
  RMSE:  48.480389
  MAE:   27.710174
  MAPE:  26.2486%
  R²:    0.812660
  [OK] LSTM - RMSE=48.48, R2=0.8127

PHASE 1 BASELINE COMPARISON SUMMARY

Model                RMSE            MAE             MAPE            R²             
--------------------------------------------------------------------------------
XGBoost              43.382231       21.877996       18.1580        % 0.849989       
LSTM                 48.480389       27.710174       26.2486        % 0.812660       
NLinear              48.858596       25.590920       18.9207        % 0.809726       
DLinear              52.179754       26.450350       20.0425        % 0.782979       

LOOKBACK PERIOD: 30 days

[AQIDataLoader] Preparing sequences (seq_len=30)...
  Train: (16837, 30, 12), Val: (2396, 30, 12), Test: (4837, 30, 12

C:\Users\Sowmithra\AppData\Local\Temp\ipykernel_7680\2248138011.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f"{col}_lag_{lag}"] = grouped[col].shift(lag)
C:\Users\Sowmithra\AppData\Local\Temp\ipykernel_7680\2248138011.py:177: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[f"{col}_rolling_mean_7"] = shifted.groupby(df_copy["City"]).transform(
C:\Users\Sowmithra\AppData\Local\Temp\ipykernel_7680\2248138011.py:180: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `f

  Features created: 400
  Train: (16837, 384), Val: (2396, 384), Test: (4837, 384)

[DLinear] lookback=30...
[DLinear] Model initialized on cpu
  Input: (batch, 30, 12)
  Output: (batch, 1, 1)

[DLinear] Training for 100 epochs...
  Epoch 10/100: Train Loss=0.002505, Val Loss=0.001282
  Epoch 20/100: Train Loss=0.001597, Val Loss=0.000817
  Epoch 30/100: Train Loss=0.001370, Val Loss=0.000698
  Epoch 40/100: Train Loss=0.001359, Val Loss=0.000667
  Epoch 50/100: Train Loss=0.001360, Val Loss=0.000644
  Epoch 60/100: Train Loss=0.001360, Val Loss=0.000627
  Epoch 70/100: Train Loss=0.001360, Val Loss=0.000616
  Epoch 80/100: Train Loss=0.001361, Val Loss=0.000607
  Epoch 90/100: Train Loss=0.001361, Val Loss=0.000602
  Epoch 100/100: Train Loss=0.001361, Val Loss=0.000598
[DLinear] Training complete. Best val loss: 0.000598

[DLinear] Evaluation Results:
  RMSE:  54.213613
  MAE:   26.676763
  MAPE:  19.9916%
  R²:    0.765406

[DLinear] Evaluation Results:
  RMSE:  54.213613
  MAE:   2

c:\Users\Sowmithra\Documents\literature-review\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[LSTM] Training complete. Best epoch: 6
  Best val_loss: 0.000640

[LSTM] Evaluation Results:
  RMSE:  55.399456
  MAE:   34.161664
  MAPE:  33.7943%
  R²:    0.755031

[LSTM] Evaluation Results:
  RMSE:  55.399456
  MAE:   34.161664
  MAPE:  33.7943%
  R²:    0.755031
  [OK] LSTM - RMSE=55.40, R2=0.7550

PHASE 1 BASELINE COMPARISON SUMMARY

Model                RMSE            MAE             MAPE            R²             
--------------------------------------------------------------------------------
XGBoost              42.474635       22.281032       18.6303        % 0.856001       
NLinear              48.578098       25.300050       18.7225        % 0.811643       
DLinear              54.213613       26.676763       19.9916        % 0.765406       
LSTM                 55.399456       34.161664       33.7943        % 0.755031       


All lookback periods complete.


## 10. Final comparison table (mirrors `run_comparison.py` / `compare_all_refactored.py` aggregation)

---
📄 **Maps to manuscript:** **Table 2 (`tab:main_results`)** -- assembled baseline metrics (this notebook's contribution to the model-comparison table).

In [15]:
results_df = pd.DataFrame(result_rows)
results_df = results_df.rename(columns={"rmse": "RMSE", "mae": "MAE", "mape": "MAPE", "r2": "R2"})
results_df["label"] = results_df["model"] + "-LB" + results_df["lookback"].astype(str)
results_df = results_df[["label", "model", "lookback", "RMSE", "MAE", "MAPE", "R2"]]
results_df = results_df.sort_values("R2", ascending=False).reset_index(drop=True)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print("Phase 1 Colab Reproduction — All Models x Lookback Periods\n")
display(results_df)

best_row = results_df.iloc[0]
print(f"\nBest model this run: {best_row['label']}  (R2={best_row['R2']:.4f}, RMSE={best_row['RMSE']:.2f})")


Phase 1 Colab Reproduction — All Models x Lookback Periods



,label,model,lookback,RMSE,MAE,MAPE,R2
0,XGBoost-LB7,XGBoost,7,41.2911,21.7684,18.3163,0.8638
1,XGBoost-LB30,XGBoost,30,42.4746,22.2810,18.6303,0.8560
2,XGBoost-LB14,XGBoost,14,43.3822,21.8780,18.1580,0.8500
3,LSTM-LB7,LSTM,7,48.3384,29.9350,31.2232,0.8133
4,LSTM-LB14,LSTM,14,48.4804,27.7102,26.2486,0.8127
5,NLinear-LB30,NLinear,30,48.5781,25.3001,18.7225,0.8116
6,NLinear-LB14,NLinear,14,48.8586,25.5909,18.9207,0.8097
7,NLinear-LB7,NLinear,7,49.2031,25.6831,18.7882,0.8065
8,DLinear-LB7,DLinear,7,50.6340,26.1057,19.7487,0.7951
9,DLinear-LB14,DLinear,14,52.1798,26.4503,20.0425,0.7830



Best model this run: XGBoost-LB7  (R2=0.8638, RMSE=41.29)


In [16]:
# Bar chart comparison (mirrors run_comparison.py's 4-panel figure)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Phase 1 Baseline Model Comparison — Colab GPU Reproduction", fontsize=14, fontweight="bold")

labels = results_df["label"].tolist()

axes[0, 0].bar(labels, results_df["RMSE"], color="steelblue", alpha=0.8)
axes[0, 0].set_title("RMSE (Lower is Better)")
axes[0, 0].tick_params(axis="x", rotation=60)
axes[0, 0].grid(axis="y", alpha=0.3)

axes[0, 1].bar(labels, results_df["MAE"], color="coral", alpha=0.8)
axes[0, 1].set_title("MAE (Lower is Better)")
axes[0, 1].tick_params(axis="x", rotation=60)
axes[0, 1].grid(axis="y", alpha=0.3)

axes[1, 0].bar(labels, results_df["MAPE"], color="mediumseagreen", alpha=0.8)
axes[1, 0].set_title("MAPE % (Lower is Better)")
axes[1, 0].tick_params(axis="x", rotation=60)
axes[1, 0].grid(axis="y", alpha=0.3)

axes[1, 1].bar(labels, results_df["R2"], color="mediumpurple", alpha=0.8)
axes[1, 1].set_title("R2 Score (Higher is Better)")
axes[1, 1].set_ylim([0, 1])
axes[1, 1].tick_params(axis="x", rotation=60)
axes[1, 1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


C:\Users\Sowmithra\AppData\Local\Temp\ipykernel_7680\2476912527.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# Line plots: metric vs. lookback period, one line per model (mirrors compare_lookback_comprehensive.py)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models_present = results_df["model"].unique().tolist()

for metric, ax, better in [("RMSE", axes[0], "Lower"), ("MAE", axes[1], "Lower"), ("R2", axes[2], "Higher")]:
    for m in models_present:
        sub = results_df[results_df["model"] == m].sort_values("lookback")
        ax.plot(sub["lookback"], sub[metric], marker="o", label=m, linewidth=2)
    ax.set_xlabel("Lookback Period (days)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} vs Lookback ({better} is Better)")
    ax.set_xticks(LOOKBACK_PERIODS)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


C:\Users\Sowmithra\AppData\Local\Temp\ipykernel_7680\3383157524.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Optional: save results for download / later reuse
results_df.to_csv("phase1_colab_comparison_metrics.csv", index=False)
with open("phase1_colab_comparison_summary.json", "w") as f:
    json.dump(all_results, f, indent=2, default=float)
print("Saved: phase1_colab_comparison_metrics.csv, phase1_colab_comparison_summary.json")


Saved: phase1_colab_comparison_metrics.csv, phase1_colab_comparison_summary.json


## 11. Reference metrics to compare against

These are the baseline numbers reported in the manuscript's Table 2, produced by the
per-city protocol described in Section 9 (per-city chronological split, strictly-lagged
XGBoost features, train-only scalers, scalar linear heads). Use them to sanity-check the run
above.

### 11.1 Table 2 headline (lookback = 7, 20-city CPCB test set)

| Model | R² | RMSE | MAE | MAPE% |
|---|---|---|---|---|
| **XGBoost** | **0.864** | 41.29 | 21.77 | 18.32 |
| DLinear | 0.795 | 50.63 | 26.11 | 19.75 |
| NLinear | 0.807 | 49.20 | 25.68 | 18.79 |
| LSTM (Keras) | 0.813 | 48.34 | 29.94 | 31.22 |

*(The Table 2 LSTM row in the manuscript is the PyTorch LSTM from the CEEMDAN-iTransformer
notebook, R²=0.818; the Keras LSTM here, R²=0.813, is this notebook's own conventional-LSTM
baseline and lands in the same range.)*

### 11.2 All lookbacks (this notebook's comparison table)

| Model | LB7 R² | LB14 R² | LB30 R² |
|---|---|---|---|
| XGBoost | 0.864 | 0.850 | 0.856 |
| NLinear | 0.807 | 0.810 | 0.812 |
| LSTM | 0.813 | 0.813 | 0.755 |
| DLinear | 0.795 | 0.783 | 0.765 |

### 11.3 What "matching" should look like

The run is seeded, so expect the same ranking and ranges rather than digit-for-digit matches:
XGBoost (~0.86) > LSTM ≈ NLinear (~0.81) > DLinear (~0.79) at LB7; XGBoost is roughly flat
across lookback and the LSTM drops at LB30.

---
📄 **Maps to manuscript:** Cross-check only -- compares this session's numbers against the final Table 2 baseline values (no new manuscript element).